In [18]:
from typing import List
import math
from typing import Any
from collections import Counter

def entropy(class_prob: List[float]) -> float:
    """wektor klas na jakie podzielono zbior danych 
    wartosci wektora to proporcja ilosci recordow w tej klasie w stosunku do wszyskich recordow"""

    return sum([-p * math.log2(p) for p in class_prob if p>0]) # if statement, bo 0log0 = 0

assert entropy([1.0]) == 0
assert entropy([0.5, 0.5]) == 1
assert 0.81 < entropy([0.25, 0.75]) < 0.82

def class_probabilities(labels: List[Any]) -> List[float]:
    total_count = len(labels)
    return [count / total_count
            for count in Counter(labels).values()]

def data_entropy(labels: List[Any]) -> float:
    return entropy(class_probabilities(labels))

assert data_entropy(['a']) == 0 
assert data_entropy([True, False]) == 1
assert data_entropy([3, 4, 4, 4]) == entropy([0.25, 0.75])

def partition_entropy(subsets: List[List[Any]]) -> float:
    """wazona entropia podzbiorow"""
    count_S = sum(len(s_i) for s_i in subsets)
    return sum([len(s_i)/count_S*data_entropy(s_i) for s_i in subsets])

In [34]:
from typing import NamedTuple, Optional

class Candidate(NamedTuple):
    level: str
    lang: str
    tweets: bool
    phd: bool
    did_well: Optional[bool] = None  # allow unlabeled data

                  #  level     lang     tweets  phd  did_well
inputs = [Candidate('Senior', 'Java',   False, False, False),
          Candidate('Senior', 'Java',   False, True,  False),
          Candidate('Mid',    'Python', False, False, True),
          Candidate('Junior', 'Python', False, False, True),
          Candidate('Junior', 'R',      True,  False, True),
          Candidate('Junior', 'R',      True,  True,  False),
          Candidate('Mid',    'R',      True,  True,  True),
          Candidate('Senior', 'Python', False, False, False),
          Candidate('Senior', 'R',      True,  False, True),
          Candidate('Junior', 'Python', True,  False, True),
          Candidate('Senior', 'Python', True,  True,  True),
          Candidate('Mid',    'Python', False, True,  True),
          Candidate('Mid',    'Java',   True,  False, True),
          Candidate('Junior', 'Python', False, True,  False)
         ]
print(getattr(inputs[0], 'level'))

Senior


In [60]:
from typing import Dict, TypeVar
from collections import defaultdict

T = TypeVar('T')  

def partition_by(inputs: List[T], attribute: str) -> Dict[Any, List[T]]:
    """Partition the inputs into lists based on the specified attribute."""
    partitions: Dict[Any, List[T]] = defaultdict(list)
    for input in inputs:
        key = getattr(input, attribute)  
        partitions[key].append(input)    
    return partitions

partition_by(inputs, 'phd')

defaultdict(list,
            {False: [Candidate(level='Senior', lang='Java', tweets=False, phd=False, did_well=False),
              Candidate(level='Mid', lang='Python', tweets=False, phd=False, did_well=True),
              Candidate(level='Junior', lang='Python', tweets=False, phd=False, did_well=True),
              Candidate(level='Junior', lang='R', tweets=True, phd=False, did_well=True),
              Candidate(level='Senior', lang='Python', tweets=False, phd=False, did_well=False),
              Candidate(level='Senior', lang='R', tweets=True, phd=False, did_well=True),
              Candidate(level='Junior', lang='Python', tweets=True, phd=False, did_well=True),
              Candidate(level='Mid', lang='Java', tweets=True, phd=False, did_well=True)],
             True: [Candidate(level='Senior', lang='Java', tweets=False, phd=True, did_well=False),
              Candidate(level='Junior', lang='R', tweets=True, phd=True, did_well=False),
              Candidate(level='Mid', l

In [21]:
def partition_entropy_by(inputs: List[Any],
                         attribute: str,
                         label_attribute: str) -> float:
    """Compute the entropy corresponding to the given partition"""
    # partitions consist of our inputs
    partitions = partition_by(inputs, attribute)

    # but partition_entropy needs just the class labels
    labels = [[getattr(input, label_attribute) for input in partition]
              for partition in partitions.values()]

    return partition_entropy(labels)

for key in ['level','lang','tweets','phd']:
    print(key, partition_entropy_by(inputs, key, 'did_well'))

assert 0.69 < partition_entropy_by(inputs, 'level', 'did_well')  < 0.70
assert 0.86 < partition_entropy_by(inputs, 'lang', 'did_well')   < 0.87
assert 0.78 < partition_entropy_by(inputs, 'tweets', 'did_well') < 0.79
assert 0.89 < partition_entropy_by(inputs, 'phd', 'did_well')    < 0.90

level 0.6935361388961918
lang 0.8601317128547441
tweets 0.7884504573082896
phd 0.8921589282623617


In [22]:
senior_inputs = [input for input in inputs if input.level == 'Senior']

assert 0.4 == partition_entropy_by(senior_inputs, 'lang', 'did_well')
assert 0.0 == partition_entropy_by(senior_inputs, 'tweets', 'did_well')
assert 0.95 < partition_entropy_by(senior_inputs, 'phd', 'did_well') < 0.96

In [ ]:
from typing import NamedTuple, Union, Any

class Leaf(NamedTuple):
    value: Any

class Split(NamedTuple):
    attribute: str
    subtrees: dict
    default_value: Any = None

DecisionTree = Union[Leaf, Split]

hiring_tree = Split('level', {   # First, consider "level".
    
    'Junior': Split('phd', {     # if level is "Junior", next look at "phd"
        False: Leaf(True),       #   if "phd" is False, predict True
        True: Leaf(False)        #   if "phd" is True, predict False
    }),
    
    'Mid': Leaf(True),           # if level is "Mid", just predict True
    
    'Senior': Split('tweets', {  # if level is "Senior", look at "tweets"
        False: Leaf(False),      #   if "tweets" is False, predict False
        True: Leaf(True)         #   if "tweets" is True, predict True
    })
})

print(hiring_tree.attribute)
print(hiring_tree.subtrees)
print(hiring_tree.subtrees['Junior'])
print(hiring_tree.subtrees['Junior'].attribute)
print(hiring_tree.subtrees['Junior'].subtrees[False])

def classify(tree: DecisionTree, input: Any) -> Any:
    """przechodzi przez drzewo i srawdza jaka wartosc jest na lisciu dla zadanego inputa"""
    
    if isinstance(tree, Leaf): # warunek koncowy rekurencji
        return tree.value
    
    attr = tree.attribute
    input_label = getattr(input, attr)
    
    if input_label not in tree.subtrees:
        return tree.default_value
    
    sub_tree = tree.subtrees[input_label]
    return classify(sub_tree, input)

assert not classify(hiring_tree, inputs[0])
assert classify(hiring_tree, Candidate('intern','Java', False, False, False)) == None

level
{'Junior': Split(attribute='phd', subtrees={False: Leaf(value=True), True: Leaf(value=False)}, default_value=None), 'Mid': Leaf(value=True), 'Senior': Split(attribute='tweets', subtrees={False: Leaf(value=False), True: Leaf(value=True)}, default_value=None)}
Split(attribute='phd', subtrees={False: Leaf(value=True), True: Leaf(value=False)}, default_value=None)
phd
Leaf(value=True)


In [ ]:
test_tree = Split('level', {})
test_tree.subtrees['senior'] = True
test_tree

Split(attribute='level', subtrees={'senior': True}, default_value=None)

In [62]:
from collections import Counter

def build_tree_id3(inputs: List[Any], split_attributes: List[str], target_attribute: str) -> DecisionTree:
    # zaczynamy od wanunkow konczycych rekurencje: 
    # nie ma wiecej atrybytow na ktore mozna podzielic -> leaf z most common
    # wszystkie labele target attribute sa takie same -> leaf z tym label
    label_count = Counter([getattr(row, target_attribute) for row in inputs])
    most_common_label = label_count.most_common(1)[0][0]    
    if not split_attributes or len(label_count) == 1: 
        return Leaf(most_common_label)
    
    min_entropy_attr = sorted([(attribute, partition_entropy_by(inputs, attribute, target_attribute)) 
                               for attribute in split_attributes], key=lambda x: x[1])[0][0]
    
    partitions = partition_by(inputs, min_entropy_attr)
    new_attributes = [attr for attr in split_attributes if attr != target_attribute]
    
    subtrees = {attr_val: build_tree_id3(subset,
                                       new_attributes,
                                       target_attribute) 
                                       for attr_val, subset in partitions.items()}
    
    return Split(min_entropy_attr, subtrees, default_value=most_common_label)
    

In [63]:
tree = build_tree_id3(inputs,
                      ['level', 'lang', 'tweets', 'phd'],
                      'did_well')

# Should predict True
assert classify(tree, Candidate("Junior", "Java", True, False))

# Should predict False
assert not classify(tree, Candidate("Junior", "Java", True, True))

# Should predict True
assert classify(tree, Candidate("Intern", "Java", True, True))